# Text Summarization with Transformer

Transformer encoder-decoder for text summarization. 

- How to apply encoder-decoder to summarization
- Handling longer input sequences
- **Beam Search** for better summary generation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import string

## 1. Dataset: News Articles and Summaries

We create a small dataset of news articles with their summaries.

Each article is 80-120 words and each summary is 15-25 words.

In [ ]:
# Article-summary pairs
data = [
    (
        "Scientists at the University of California have made a groundbreaking discovery in renewable energy. The new solar panel design increases efficiency by forty percent compared to traditional models. This breakthrough could revolutionize the clean energy industry and significantly reduce costs for consumers. The research team spent five years developing the technology using advanced materials. Initial tests show promising results and commercial production may begin next year.",
        "UC scientists develop new solar panels with forty percent higher efficiency potentially revolutionizing clean energy industry"
    ),
    (
        "A major technology company announced plans to invest ten billion dollars in artificial intelligence research over the next decade. The initiative aims to advance machine learning capabilities and develop new applications across various industries. Company executives stated that AI will transform how businesses operate and improve customer experiences. The investment will fund research facilities and hire thousands of AI specialists worldwide.",
        "Tech company announces ten billion dollar investment in AI research to advance machine learning and transform business operations"
    ),
    (
        "Education experts recommend increasing investment in early childhood programs to improve long-term academic outcomes. Studies show that children who attend quality preschool programs perform better throughout their educational careers. The programs focus on developing cognitive and social skills during critical developmental years. Governments are considering expanding funding to make these programs more accessible. Research indicates significant returns on investment in early education.",
        "Experts recommend investing in early childhood education programs showing improved long-term academic outcomes"
    ),
    (
        "Space agencies are planning a collaborative mission to establish a permanent research station on the Moon within the next decade. The facility will support scientific experiments and serve as a stepping stone for future Mars missions. Advanced life support systems will enable astronauts to live and work on the lunar surface for extended periods. International cooperation is essential for the ambitious project. Construction could begin as early as twenty twenty-eight.",
        "Space agencies plan permanent Moon research station within decade as stepping stone to Mars missions"
    ),
    (
        "Agricultural scientists have developed drought-resistant crop varieties that maintain high yields even in water-scarce conditions. The breakthrough could help address food security challenges in regions affected by climate change. Traditional breeding techniques combined with modern genetic tools produced the resilient plants. Farmers in arid areas are eager to adopt the new varieties. Field tests show promising results across multiple growing seasons.",
        "Scientists develop drought-resistant crops maintaining high yields to address food security in water-scarce regions"
    ),
    (
        "Cybersecurity experts warn that ransomware attacks have increased by sixty percent over the past year targeting businesses and government institutions. The sophisticated attacks encrypt critical data and demand payment for its release. Organizations are urged to implement robust security measures and employee training programs. Regular data backups and updated software are essential defenses. Law enforcement agencies are working internationally to combat cybercrime networks.",
        "Ransomware attacks surge sixty percent targeting businesses and governments prompting calls for stronger cybersecurity measures"
    ),
    (
        "Wildlife conservation efforts have successfully increased endangered tiger populations by thirty percent over the past decade. Protected habitats and anti-poaching measures contributed to the recovery. International cooperation between governments and conservation organizations proved essential. Local communities play a vital role in protecting wildlife through sustainable practices. The success demonstrates that targeted conservation strategies can reverse population declines.",
        "Conservation efforts increase endangered tiger populations thirty percent through habitat protection and anti-poaching measures"
    ),
    (
        "Neuroscientists made significant progress in understanding how the brain processes and stores memories. Advanced imaging techniques revealed previously unknown neural pathways involved in memory formation. The findings could lead to new treatments for conditions like Alzheimer's disease and other memory disorders. Research continues to explore the complex mechanisms underlying human cognition. The discoveries open new avenues for therapeutic interventions.",
        "Neuroscientists discover new neural pathways in memory formation potentially leading to Alzheimer's treatments"
    )
]

print(f"Dataset size: {len(data)} article-summary pairs")

## 2. Building Vocabularies

We need separate vocabularies for articles (source) and summaries (target).

In [ ]:
def build_vocabulary(texts):
    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
    for text in texts:
        text = text.translate(str.maketrans('', '', string.punctuation)).lower()
        for word in text.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

articles = [article for article, _ in data]
summaries = [summary for _, summary in data]

source_vocab = build_vocabulary(articles)
target_vocab = build_vocabulary(summaries)

source_inverse_vocab = {idx: word for word, idx in source_vocab.items()}
target_inverse_vocab = {idx: word for word, idx in target_vocab.items()}

print(f"Source vocabulary size: {len(source_vocab)}")
print(f"Target vocabulary size: {len(target_vocab)}")

## 3. Text Preprocessing

Convert text to token indices.

In [ ]:
def encode_text(text, vocab):
    text = text.translate(str.maketrans('', '', string.punctuation)).lower()
    return [vocab.get(word, vocab["<UNK>"]) for word in text.split()]

def decode_text(indices, inverse_vocab):
    return " ".join([inverse_vocab[idx] for idx in indices if idx > 2])

## 4. Positional Encoding

Adds position information to embeddings.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_length=200):
        super().__init__()
        position_encoding = torch.zeros(max_length, embedding_dim)
        position = torch.arange(0, max_length).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-math.log(10000.0) / embedding_dim))
        
        position_encoding[:, 0::2] = torch.sin(position * div_term)
        position_encoding[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('position_encoding', position_encoding.unsqueeze(0))
    
    def forward(self, x):
        return x + self.position_encoding[:, :x.size(1)]

## 5. Summarization Model

We use PyTorch's built-in Transformer for the encoder-decoder architecture.

In [ ]:
class SummarizationTransformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, embedding_dim=128, num_heads=4, 
                 num_encoder_layers=2, num_decoder_layers=2, feedforward_dim=256, dropout=0.1):
        super().__init__()
        
        self.source_embedding = nn.Embedding(source_vocab_size, embedding_dim)
        self.target_embedding = nn.Embedding(target_vocab_size, embedding_dim)
        self.positional_encoding = PositionalEncoding(embedding_dim)
        
        self.transformer = nn.Transformer(
            d_model=embedding_dim,
            nhead=num_heads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True
        )
        
        self.output_projection = nn.Linear(embedding_dim, target_vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def create_target_mask(self, target_length):
        """Create a mask for the target sequence to prevent attention to future tokens."""
        mask = torch.triu(torch.ones(target_length, target_length), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, source, target):
        target_mask = self.create_target_mask(target.size(1)).to(source.device)
        source_padding_mask = (source == 0)
        target_padding_mask = (target == 0)
        
        source_embedded = self.dropout(self.positional_encoding(self.source_embedding(source)))
        target_embedded = self.dropout(self.positional_encoding(self.target_embedding(target)))
        
        output = self.transformer(
            source_embedded,
            target_embedded,
            tgt_mask=target_mask,
            src_key_padding_mask=source_padding_mask,
            tgt_key_padding_mask=target_padding_mask
        )
        
        return self.output_projection(output)

## 6. Training Setup

We prepare input/output pairs and initialize the model.

In [ ]:
def prepare_training_data(data, source_vocab, target_vocab):
    pairs = []
    for article, summary in data:
        source = torch.tensor(encode_text(article, source_vocab), dtype=torch.long)
        target_input = torch.tensor([target_vocab["<SOS>"]] + encode_text(summary, target_vocab), dtype=torch.long)
        target_output = torch.tensor(encode_text(summary, target_vocab) + [target_vocab["<EOS>"]], dtype=torch.long)
        pairs.append((source, target_input, target_output))
    return pairs

training_data = prepare_training_data(data, source_vocab, target_vocab)

model = SummarizationTransformer(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab)
)

optimizer = optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CrossEntropyLoss(ignore_index=0)

## 7. Training

In [ ]:
num_epochs = 400
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for source, target_input, target_output in training_data:
        source, target_input, target_output = source.unsqueeze(0), target_input.unsqueeze(0), target_output.unsqueeze(0)
        optimizer.zero_grad()
        output = model(source, target_input)
        loss = criterion(output.view(-1, len(target_vocab)), target_output.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(training_data):.4f}")

## 8. Accurate Decoding (Beam Search)

**Beam Search** explores multiple possible paths to find the most accurate summary.
Beam search aims to find the most likely sequence of tokens that the model would generate.

In [ ]:
def generate_summary(article, model, source_vocab, target_vocab, target_inverse_vocab, beam_width=3, max_length=50):
    model.eval()
    source = torch.tensor(encode_text(article, source_vocab), dtype=torch.long).unsqueeze(0)
    
    # beams: list of (sequence, score)
    beams = [([target_vocab["<SOS>"]], 0.0)]
    
    for _ in range(max_length):
        new_beams = []
        for seq, score in beams:
            if seq[-1] == target_vocab["<EOS>"]:
                new_beams.append((seq, score))
                continue
            
            target = torch.tensor([seq], dtype=torch.long)
            with torch.no_grad():
                output = model(source, target)
            
            probs = F.log_softmax(output[:, -1, :], dim=-1)[0]
            top_probs, top_indices = probs.topk(beam_width)
            
            for i in range(beam_width):
                new_beams.append((seq + [top_indices[i].item()], score + top_probs[i].item()))
        
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        if all(seq[-1] == target_vocab["<EOS>"] for seq, _ in beams): break
            
    return decode_text(beams[0][0], target_inverse_vocab)

## 9. Testing on New/Unseen Text

TASK: Test the model on new/unseen text

Plan new test articles and see if the model can generate a summary.

Where are we lacking or not doing well?